In [1]:
import nemo.collections.asr as nemo_asr

c:\Users\sidef\Test Audio\aud\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[NeMo W 2026-04-08 16:23:39 megatron_init:62] Megatron num_microbatches_calculator not found, using Apex version.
W0408 16:23:39.402000 37696 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
OneLogger: Setting error_handling_strategy to DISABLE_QUIETLY_AND_REPORT_METRIC_ERROR for rank (rank=0) with OneLogger disabled. To override: explicitly set error_handling_strategy parameter.
No exporters were provided. This means that no telemetry data will be collected.
W0408 16:23:42.044000 37696 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


In [5]:
import torch
import time
import pandas as pd
import os
from pydub import AudioSegment
from tqdm import tqdm

In [3]:
asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name="nvidia/parakeet-tdt-0.6b-v3")

[NeMo I 2026-04-08 16:24:13 mixins:184] Tokenizer SentencePieceTokenizer initialized with 8192 tokens


[NeMo W 2026-04-08 16:24:18 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    use_lhotse: true
    skip_missing_manifest_entries: true
    input_cfg: null
    tarred_audio_filepaths: null
    manifest_filepath: null
    sample_rate: 16000
    shuffle: true
    num_workers: 2
    pin_memory: true
    max_duration: 10.0
    min_duration: 1.0
    text_field: answer
    batch_duration: null
    max_tps: null
    use_bucketing: true
    bucket_duration_bins: null
    bucket_batch_size: null
    num_buckets: 30
    bucket_buffer_size: 20000
    shuffle_buffer_size: 10000
    
[NeMo W 2026-04-08 16:24:18 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the validation data loader(s). 
    Validation 

[NeMo I 2026-04-08 16:24:27 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-04-08 16:24:27 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-04-08 16:24:27 rnnt_models:226] Using RNNT Loss : tdt
    Loss tdt_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0, 'durations': [0, 1, 2, 3, 4], 'sigma': 0.02, 'omega': 0.1}
[NeMo I 2026-04-08 16:24:30 save_restore_connector:285] Model EncDecRNNTBPEModel was successfully restored from C:\Users\sidef\.cache\huggingface\hub\models--nvidia--parakeet-tdt-0.6b-v3\snapshots\6d590f77001d318fb17a0b5bf7ee329a91b52598\parakeet-tdt-0.6b-v3.nemo.


In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"
asr_model = asr_model.to(device)

asr_model.eval()

print(f"Model loaded on: {device}")

Model loaded on: cuda


In [4]:
tsv_path = "labels/ss-corpus-en.tsv"
df = pd.read_csv(tsv_path, sep="\t")

print("Columns:", df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())

print("\nChecking 'transcription' column:")
if "transcription" in df.columns:
    print(df["transcription"].head())
else:
    print("No 'transcription' column found. Available columns:", df.columns.tolist())

Columns: ['client_id', 'audio_id', 'audio_file', 'duration_ms', 'prompt_id', 'prompt', 'transcription', 'votes', 'age', 'gender', 'accents', 'variant', 'language', 'prompt_upvotes', 'prompt_reports', 'is_edited', 'split', 'char_per_sec', 'quality_tags']

First 5 rows:


,client_id,audio_id,audio_file,duration_ms,prompt_id,prompt,transcription,votes,age,gender,accents,variant,language,prompt_upvotes,prompt_reports,is_edited,split,char_per_sec,quality_tags
0,3cf166c6b73f030b4f67eeaeba301103,1,spontaneous-speech-en-1.mp3,10044,152,How do you take care of your body and health?,"I workout a lot, running, stretching, and taki...",1,NaN,NaN,NaN,NaN,English,0,0,0,test,8.363202,NaN
1,3cf166c6b73f030b4f67eeaeba301103,2,spontaneous-speech-en-2.mp3,11880,142,What are you proud of?,"I'm really proud of my wife, she's quite a fan...",1,NaN,NaN,NaN,NaN,English,0,0,1,test,10.353535,NaN
2,3cf166c6b73f030b4f67eeaeba301103,3,spontaneous-speech-en-3.mp3,6444,226,Who is your favourite celebrity or actor?,"for whatever reason, I enjoy watching Nicolas ...",1,NaN,NaN,NaN,NaN,English,0,0,0,test,6.672874,NaN
3,12b668a1ada1828ba795332f419d4ef7,4,spontaneous-speech-en-4.mp3,5148,138,When you were a child what did you want to be ...,Told my teachers I wanted to be my own boss wh...,1,NaN,NaN,NaN,NaN,English,0,0,1,test,8.741259,NaN
4,12b668a1ada1828ba795332f419d4ef7,5,spontaneous-speech-en-5.mp3,5436,121,What do you want to do with your next day off?,"I wanna go hiking, in the woods",1,NaN,NaN,NaN,NaN,English,0,0,0,test,4.598970,NaN



Checking 'transcription' column:
0    I workout a lot, running, stretching, and taki...
1    I'm really proud of my wife, she's quite a fan...
2    for whatever reason, I enjoy watching Nicolas ...
3    Told my teachers I wanted to be my own boss wh...
4                      I wanna go hiking, in the woods
Name: transcription, dtype: str


In [7]:
audio_dir = "wav_audio"

df["audio_path"] = df["audio_file"].str.replace(".mp3", ".wav", regex=False)
df["audio_path"] = df["audio_path"].apply(lambda x: os.path.join(audio_dir, x))

print(df[["audio_file", "audio_path"]].head())

                    audio_file                             audio_path
0  spontaneous-speech-en-1.mp3  wav_audio\spontaneous-speech-en-1.wav
1  spontaneous-speech-en-2.mp3  wav_audio\spontaneous-speech-en-2.wav
2  spontaneous-speech-en-3.mp3  wav_audio\spontaneous-speech-en-3.wav
3  spontaneous-speech-en-4.mp3  wav_audio\spontaneous-speech-en-4.wav
4  spontaneous-speech-en-5.mp3  wav_audio\spontaneous-speech-en-5.wav


In [8]:
print(df["audio_path"].apply(os.path.exists).value_counts())

audio_path
False    6378
True        4
Name: count, dtype: int64


In [10]:
df = df[df["audio_path"].apply(lambda x: os.path.exists(x))].reset_index(drop=True)

print("Remaining files:", len(df))
print(df["audio_path"].tolist())

Remaining files: 4
['wav_audio\\spontaneous-speech-en-1.wav', 'wav_audio\\spontaneous-speech-en-2.wav', 'wav_audio\\spontaneous-speech-en-3.wav', 'wav_audio\\spontaneous-speech-en-4.wav']


In [11]:
predictions = []
latencies = []

for audio_path in tqdm(df["audio_path"]):
    start_time = time.time()
    
    # Run inference
    pred = asr_model.transcribe([audio_path])[0]
    
    end_time = time.time()
    
    predictions.append(pred)
    latencies.append(end_time - start_time)

# Add results to dataframe
df["prediction"] = predictions
df["latency_sec"] = latencies

print("Inference completed!")
df[["audio_path", "prediction", "latency_sec"]].head()

  0%|          | 0/4 [00:00<?, ?it/s][NeMo W 2026-04-08 16:36:04 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-04-08 16:36:04 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:00,  3.49it/s]
 25%|██▌       | 1/4 [00:00<00:00,  3.15it/s][NeMo W 2026-04-08 16:36:04 dataloader:826] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-04-08 16:36:04 dataloader:523] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to 

Inference completed!


,audio_path,prediction,latency_sec
0,wav_audio\spontaneous-speech-en-1.wav,"Hypothesis(score=-135.9397430419922, y_sequenc...",0.317230
1,wav_audio\spontaneous-speech-en-2.wav,"Hypothesis(score=-135.2339324951172, y_sequenc...",0.098603
2,wav_audio\spontaneous-speech-en-3.wav,"Hypothesis(score=-53.352622985839844, y_sequen...",0.080160
3,wav_audio\spontaneous-speech-en-4.wav,"Hypothesis(score=-82.7293701171875, y_sequence...",0.085992


In [13]:
avg_latency = df["latency_sec"].mean()
avg_latency

np.float64(0.1454961895942688)

In [14]:
from jiwer import wer, cer, Compose, ToLowerCase, RemovePunctuation, Strip

In [19]:
df["prediction"] = df["prediction"].apply(lambda x: x.text if hasattr(x, "text") else x)

# Normalization pipeline
transform = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    Strip()
])

wers = []
cers = []

filtered_gt = []
filtered_pred = []

for gt, pred in zip(df["transcription"], df["prediction"]):
    # Normalize
    gt_clean = transform(gt)
    pred_clean = transform(pred)
    
    # Skip empty ground truth
    if len(gt_clean.strip()) == 0:
        print("Skipping empty ground truth")
        continue
    
    w = wer(gt_clean, pred_clean)
    c = cer(gt_clean, pred_clean)
    
    wers.append(w)
    cers.append(c)
    filtered_gt.append(gt)
    filtered_pred.append(pred)

# Print per-sample results
for i, (gt, pred, w, c) in enumerate(zip(filtered_gt, filtered_pred, wers, cers)):
    print(f"\nSample {i+1}:")
    print("GT :", gt)
    print("Pred:", pred)
    print(f"WER: {w:.4f} | CER: {c:.4f}")

# Final averages
avg_wer = sum(wers) / len(wers)
avg_cer = sum(cers) / len(cers)

print("\n===== FINAL RESULTS =====")
print(f"Average WER: {avg_wer:.4f}")
print(f"Average CER: {avg_cer:.4f}")


Sample 1:
GT : I workout a lot, running, stretching, and taking care of myself through <disfluency> a healthy diet
Pred: I work out a lot running, stretching, and taking care of myself through a healthy diet.
WER: 0.1875 | CER: 0.1458

Sample 2:
GT : I'm really proud of my wife, she's quite a fantastic person; <disfluency> inspiring, and just <disfluency> full of a really positive a good energy
Pred: I'm really proud of my wife. She's quite a fantastic person, uh inspiring and just uh full of uh really positive uh good energy.
WER: 0.1667 | CER: 0.1844

Sample 3:
GT : for whatever reason, I enjoy watching Nicolas Cage
Pred: For whatever reason, I enjoy watching Nicolas Cage.
WER: 0.0000 | CER: 0.0000

Sample 4:
GT : Told my teachers I wanted to be my own boss when I grew up
Pred: I told my teachers I wanted to be my own boss when I grew up.
WER: 0.0714 | CER: 0.0345

===== FINAL RESULTS =====
Average WER: 0.1064
Average CER: 0.0912
